## RUN - 1 - Preprocessing

### Notes

- **Information on input**
    - `\ex_vivo\Ectoderm_RFP_Spots`; ectoderm cell tracks with track ID, x/y positions (in microns), and time (in frames, not sorted)
    - `\ex_vivo\Myeloid_FDX_Tracks`; myeloid cell tracks with track ID (TID), x/y positions (in microns), and time (in sec, sorted)
    - Each `Pos(ddd)` is an explant; they should match between ectoderm and myeloid
    - `norm` means normal fibronectin, `dilute` / `dilu` means low fibronectin
    - Tracking was done by Anh with StarDist for the ectoderm and manually in ImageJ for myeloid cells
    - Spatial resolution is `1px = 1.5152um` and temporal resolution is `1tp = 300sec = 5min`


* **Content of this notebook**
    1. Parse and assemble ectoderm data
    2. Parse and assemble myeloid data
    3. Remove short tracks from ectoderm data
    4. Ensure all myeloid time points have ectoderm data
    5. Save prepped data for next steps
    

- **Outputs of this notebook**
    - `ecto_data[conditions][positions]`; df of shape `tracks X (f, t, y, x)`
    - `myel_data[conditions][positions]`; df of shape `tracks X (f, t, y, x)`
    - with units: `[f] = [1]`, `[t] = [min]`, `[y, x] = [microns]`

### Prep

In [ ]:
### Imports

import os, warnings, pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from ipywidgets import interact

import sys; sys.path.insert(0, '..')
from tracking_analysis import preprocessing as prep
from tracking_analysis.utilities import savebutton

In [ ]:
### Parameters

# Overwrite or dry run
save_outputs = True

# Units
pxl_res  = 1.5152  # [microns]  # Not needed bc tracking data is now in microns already
time_res = 5       # [min]
myel_frame_time = 300.01  # [sec]  # As per times given in MTrackJ files

# Data cleaning
ecto_min_tracklen = 10

In [ ]:
### Data locations

top_path = r"..\Data\ex_vivo"

ecto_dir = r"Ectoderm_RFP_Spots"
myel_dir = r"Myeloid_FDX_Tracks"

### Parse and assemble data

In [ ]:
### Load & prep ectoderm data

# Prep data structure
ecto_data = {'norm' : {}, 'dilu' : {}}

# For each explant...
for fname in os.listdir(os.path.join(top_path, ecto_dir)):
    fpath = os.path.join(top_path, ecto_dir, fname)
    
    if fname.endswith('_spots.csv'):
        
        # Check for condition
        if ('_norm_' not in fname) and ('dilute_' not in fname):
            warnings.warn("\nFound file with unexpected condition: "+fpath+"\n")
            continue
        
        # Load the data
        _, tfyx = prep.load_stardist_pandas(fpath)
        
        # Convert frames to minutes
        tfyx.insert(1, 't', tfyx['f'] * time_res)
        
        # Add to data structure
        pos = fname.split('_')[0]
        if '_norm_' in fname:
            ecto_data['norm'][pos] = tfyx
        elif '_dilute_' in fname:
            ecto_data['dilu'][pos] = tfyx
            
    else:
        warnings.warn("\nFound unexpected file: "+fpath+"\n")
        
# Report
display(ecto_data['norm'].keys())
display(ecto_data['dilu'].keys())
display(ecto_data['norm']['Pos001'].head())

In [ ]:
### Load & prep myeloid data

# Prep data structure
myel_data = {'norm' : {}, 'dilu' : {}}

# For each explant...
for fname in os.listdir(os.path.join(top_path, myel_dir)):
    fpath = os.path.join(top_path, myel_dir, fname)
    
    if fname.endswith('-Points.csv'):
        
        # Check for condition and for matching Pos in ectoderm data
        pos = fname.split('_')[0]
        if ('_norm_' not in fname) and ('dilute_' not in fname):
            warnings.warn("\nFound file with unexpected condition: "+fpath+"\n")
            continue
        elif (pos in ecto_data['norm'].keys()) and ('_norm_' in fname):
            pass
        elif (pos in ecto_data['dilu'].keys()) and ('_dilute_' in fname):
            pass
        else:
            warnings.warn("\nFound Pos number with no matching ectoderm: "+pos+"\n")
            continue
        
        # Load the data
        _, tfyx = prep.load_mtrackj_pandas(fpath, myel_frame_time)
        
        # Convert frames to minutes
        tfyx.insert(1, 't', tfyx['f'] * time_res)
        
        # Add to data structure
        if '_norm_' in fname:
            myel_data['norm'][pos] = tfyx
        elif '_dilute_' in fname:
            myel_data['dilu'][pos] = tfyx
        
    else:
        warnings.warn("\nFound unexpected file: "+fpath+"\n")
        
# Report
display(myel_data['norm'].keys())
display(myel_data['dilu'].keys())
display(myel_data['norm']['Pos001'].head())

In [ ]:
### Save the assembled raw data

if save_outputs:

    # Ectoderm
    with open(os.path.join(top_path, "Ectoderm-data.pkl"), "wb") as outfile:
        pickle.dump(ecto_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)

    # Myeloid
    with open(os.path.join(top_path, "Myeloid-data.pkl"), "wb") as outfile:
        pickle.dump(myel_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)

### Basic data cleaning

In [ ]:
### Look for & remove short tracks ("stubs")

# Check for presence of stubs
fig, ax = plt.subplots(2, 2, figsize=(7, 4), sharex=True)
_, ecto_bins, _ = ax[0,0].hist(ecto_data['norm']['Pos001'].index.value_counts(sort=False), bins=250)
_, myel_bins, _ = ax[0,1].hist(myel_data['norm']['Pos001'].index.value_counts(sort=False), bins=250)
ax[0,0].set_title('ectoderm'); ax[0,1].set_title('myeloid')
ax[0,0].set_ylabel('count'); ax[0,1].set_ylabel('count')
# ->> Myeloid doesn't have any stubs (due to manual tracking)
# ->> With the new tracking, ectoderm also doesn't have any stubs anymore

# Remove ectoderm tracks shorter than ecto_min_tracklen
for condition in ecto_data.keys():
    for position in ecto_data[condition].keys():
        ecto_tracklens = ecto_data[condition][position].index.value_counts(sort=False)
        keep_tracks = ecto_tracklens.index[ecto_tracklens >= ecto_min_tracklen]
        keep_rows = np.isin(ecto_data[condition][position].index, keep_tracks)
        ecto_data[condition][position] =  ecto_data[condition][position].loc[keep_rows, :]
        
# Show the result
ax[1,0].hist(ecto_data['norm']['Pos001'].index.value_counts(sort=False), bins=ecto_bins)
ax[1,1].hist(myel_data['norm']['Pos001'].index.value_counts(sort=False), bins=250)
ax[1,0].set_xlabel('track length'); ax[1,1].set_xlabel('track length')
ax[0,0].set_ylabel('count'); ax[1,1].set_ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
### Ensure all myeloid time points have ectoderm data

for c in ecto_data.keys():
    for p in ecto_data[c].keys():
        print(f'{c}, {p}:', end=' ')
        no_issues = True
        
        # Crop at start if myeloid starts earlier
        ecto_start = ecto_data[c][p]['f'].unique().min()
        myel_start = myel_data[c][p]['f'].unique().min()
        if ecto_start > myel_start:
            myel_mask = myel_data[c][p]['f'] >= ecto_start
            myel_data[c][p] = myel_data[c][p].loc[myel_mask, :]
            no_issues = False
            print(f"Myeloid start cropped from {myel_start} to {ecto_start}.", end=' ')
        
        # Crop at end if myeloid continues later
        ecto_end = ecto_data[c][p]['f'].unique().max()
        myel_end = myel_data[c][p]['f'].unique().max()
        if ecto_end < myel_end:
            myel_mask = myel_data[c][p]['f'] <= ecto_end
            myel_data[c][p] = myel_data[c][p].loc[myel_mask, :]
            no_issues = False
            print(f"Myeloid end cropped from {myel_end} to {ecto_end}.", end=' ')
        
        # Checks
        ecto_frames = np.sort(ecto_data[c][p]['f'].unique())
        myel_frames = np.sort(myel_data[c][p]['f'].unique())        
        assert np.all(ecto_frames==np.arange(ecto_frames.min(), ecto_frames.max()+1))  # Is ectoderm continuous?
        assert np.all(np.isin(myel_frames, ecto_frames))  # Do all myeloid tps have ectoderm data?
        if no_issues:
            print('pass')
        else:
            print()

### Save the results

In [ ]:
### Save the cleaned data

if save_outputs:

    # Ectoderm
    with open(os.path.join(top_path, "Ectoderm-data_clean.pkl"), "wb") as outfile:
        pickle.dump(ecto_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)

    # Myeloid
    with open(os.path.join(top_path, "Myeloid-data_clean.pkl"), "wb") as outfile:
        pickle.dump(myel_data, outfile, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
### Visualize the cleaned data

@interact(condition=ecto_data.keys())
def show_by_condition(condition='norm'):
    
    @interact(position=ecto_data[condition].keys(),
              ecto_fraction=['10%', '0%', '5%', '10%', '20%'],
              show_overlay=True)
    @savebutton
    def show_by_position(position=list(ecto_data[condition].keys())[0],
                         ecto_fraction='10%',
                         show_overlay=True):
        
        # Select data
        ecto_df = ecto_data[condition][position]
        myel_df = myel_data[condition][position]
        
        # Handle overlay vs separate subplots
        if show_overlay:
            fig, ax = plt.subplots(1, figsize=(7, 7))
            ax = [ax, ax]
        else:
            fig, ax = plt.subplots(1, 2, figsize=(10, 5))
            
        # Plot myeloid tracks
        for TID in myel_df.index.unique():
            ax[1].scatter(
                myel_df.loc[myel_df.index==TID, 'x'],
                myel_df.loc[myel_df.index==TID, 'y'],
                c=myel_df.loc[myel_df.index==TID, 'f'],
                cmap='autumn_r', s=15)
        
        # Plot ectoderm tracks
        if ecto_fraction != '0%':
            for TID in ecto_df.index.unique()[::100//int(ecto_fraction[:-1])]:
                ax[0].scatter(
                    ecto_df.loc[ecto_df.index==TID, 'x'],
                    ecto_df.loc[ecto_df.index==TID, 'y'],
                    c=ecto_df.loc[ecto_df.index==TID, 'f'],
                    cmap='winter_r', s=5, alpha=0.3)
        
        # Cosmetics
        for axis in ax:
            axis.axis('equal')
            axis.axis('off')
        
        # Finalize
        plt.tight_layout()